### Basic Broiler plate

In [1]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

### 1. What is an Agent?

In [ ]:
# An agent combines:

# LLM
# +
# Tools
# +
# Decision making
# +
# Loop / state



# A normal chain follows a predefined path:

# Input
#  ↓
# Prompt
#  ↓
# Gemini
#  ↓
# Output




# An agent is different.

# It can decide:

# "What should I do next?"




# For example:

# User
#  ↓
# Agent
#  ↓
# Gemini
#  ↓
# Should I use a tool?
#  ↓
# Yes
#  ↓
# Tool
#  ↓
# Tool result
#  ↓
# Gemini
#  ↓
# Do I need another tool?
#  ↓
# No
#  ↓
# Final answer




# Simple definition -

# An agent is an LLM-powered system that can dynamically choose and use tools to complete a task.

### 2. Modern LangChain Agent

In [3]:
# For modern LangChain projects, use:

from langchain.agents import create_agent



from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str: 
    """Get the weather for a city."""   # Tool description (Important to have to get tool info)
    return f"The weather in {city} is sunny."




# The basic modern structure is:

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant."
)




# Then invoke it:

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What's the weather in Bhubaneswar?"
        }
    ]
})



# The agent result contains the message/state information.

# For a simple example:

print(result["messages"][-1].text)

The weather in Bhubaneswar is currently sunny.


### 3. Complete First Agent Example

In [ ]:
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_agent

load_dotenv()


# Model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)


# Tool
@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny."


# Agent
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant."
)


# Invoke
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What's the weather in Bhubaneswar?"
        }
    ]
})


# Final response
print(result["messages"][-1].text)




# Flow

# User
#  ↓
# Agent
#  ↓
# Gemini
#  ↓
# get_weather()
#  ↓
# Tool result
#  ↓
# Gemini
#  ↓
# Final answer

The weather in Bhubaneswar is currently sunny.


### 4. What actually happens inside the Agent?

In [ ]:
# This is the most important part to understand.



# Suppose the user says:
# What's the weather in Bhubaneswar?



# The agent doesn't simply do:
# Prompt → Gemini → Answer



# Instead, conceptually:

#                     User
#                       ↓
#                     Agent
#                       ↓
#                     Gemini
#                       ↓
#               "I need weather."
#                       ↓
#               get_weather(...)
#                       ↓
#                Tool executes
#                       ↓
#               Tool result
#                       ↓
#                     Gemini
#                       ↓
#                 Final answer

# The agent manages this interaction.

### 5. Agent Loop with multiple tools

In [ ]:
# The agent's core idea is a loop:

# User
#  ↓
# Model
#  ↓
# Decision
#  ↓
# Tool
#  ↓
# Observation / result
#  ↓
# Model
#  ↓
# Decision
#  ↓
# Tool?
#  ↓
# ...
#  ↓
# Final answer





# For example:



# User:
# What's the weather in Bhubaneswar and calculate 125 × 12?
#        ↓
# Gemini:
# I need weather information.
#        ↓
# get_weather("Bhubaneswar")
#        ↓
# Weather result
#        ↓
# Gemini:
# I also need a calculation.
#        ↓
# calculator("125 * 12")
#        ↓
# Calculator result
#        ↓
# Gemini
#        ↓
# Final answer


# This loop is what makes an agent different from a simple chain.

In [6]:
from langchain_core.tools import tool
from langchain.agents import create_agent


@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"It's sunny in {city}."


@tool
def calculate(expression: str) -> str:
    """Calculate a mathematical expression."""
    # Demo implementation only.
    # In production, use a safe math parser instead of eval().
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception:
        return "Invalid mathematical expression."


tools = [
    get_weather,
    calculate
]


agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant."
)

# Now the agent has:

#              Gemini
#                 │
#         ┌───────┴────────┐
#         ↓                ↓
#  get_weather          calculate


In [7]:
# Weather Agent

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What's the weather in Bhubaneswar?"
        }
    ]
})
print(result["messages"][-1].text)

It's currently sunny in Bhubaneswar.


In [8]:
# Calculation Agent

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Calculate 125 * 12."
        }
    ]
})

print(result["messages"][-1].text)

125 * 12 = 1500


In [9]:
# Both Agents
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "What's the weather in Bhubaneswar "
                "and calculate 125 * 12."
            )
        }
    ]
})

print(result["messages"][-1].text)

The weather in Bhubaneswar is currently sunny, and 125 * 12 is 1500.


In [13]:
# Both Agents responce streeming
for chunk in agent.stream({
    "messages": [
        {
            "role": "user",
            "content": (
                "What's the weather in Bhubaneswar "
                "and calculate 125 * 12."
            )
        }
    ]
}):
    print(chunk)

{'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'calculate', 'arguments': '{"expression": "125 * 12"}'}, '__gemini_function_call_thought_signatures__': {'call_1171980': 'El4KXAFpFH0T5yzKVgK3uGInf1PFRNPnqe4VpLemco1BzDavgWf8TEbp/fXvWQci2kzcgpAnaI/ZqN6vTe7XNT7J1UMOkyQMmq21+VDOAse4Y4v8uDxHt6CAtEu8Mloa'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0bab4-c50a-7751-9202-621c4acae95b-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Bhubaneswar'}, 'id': 'call_1171980', 'type': 'tool_call'}, {'name': 'calculate', 'args': {'expression': '125 * 12'}, 'id': 'call_1171981', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 109, 'output_tokens': 38, 'total_tokens': 147, 'input_token_details': {'cache_read': 0}})]}}
{'tools': {'messages': [ToolMessage(content="It's sunny in Bhubaneswar.", name='get_weather', id=

In [ ]:

# The important thing is that you didn't manually select the tool.

# You gave the agent:
# tools = [get_weather, calculate]

# and the model can decide which tool(s) are relevant.

### 6. Agent State

In [ ]:
# An agent needs to keep track of what has happened.




# For example:

# User message
#  ↓
# Model response
#  ↓
# Tool call
#  ↓
# Tool result
#  ↓
# Model response

# These messages form part of the agent's state.




# Conceptually:

# state
# ├── user message
# ├── AI message
# ├── tool call
# ├── tool result
# └── final AI message




# That's why agent results contain:

# result["messages"]



# You can inspect them:

# for message in result["messages"]:
#     print(type(message).__name__)
#     print(message)
#     print("-" * 50)



# This is very useful for learning what's actually happening inside the agent.

### 7. Conversation History

In [16]:
# An agent can work with a sequence of messages.


# For example:

messages = [
    {
        "role": "user",
        "content": "My name is Alok."
    }
]

# Then:

result = agent.invoke({
    "messages": messages
})

print(result["messages"][-1].text)

# The important concept is:

# Conversation history is part of the agent's state.

# For an actual chatbot, the application maintains the conversation state and sends 
# the relevant messages/state to the agent on subsequent turns.

# Conceptually:

# Turn 1
# User → Agent → AI

# Turn 2
# User → Agent → AI

# Turn 3
# User → Agent → AI

Hello Alok! How can I help you today?


### 8. Agent Streaming with Updates

In [ ]:
# Normal LLM streaming gives you generated text progressively.



# Agent streaming is more interesting because an agent can have several events:

# Model
#  ↓
# Tool call
#  ↓
# Tool result
#  ↓
# Model
#  ↓
# Final answer




# Basic usage:

for chunk in agent.stream({
    "messages": [
        {
            "role": "user",
            "content": "What's the weather in Bhubaneswar?"
        }
    ]
}):
    print(chunk)



# Don't try to memorize every event structure yet.

# First understand:

# LLM streaming
#       ↓
# Chain streaming
#       ↓
# Agent streaming

# Agent streaming can expose the execution process, not merely the final text.






# A useful mental model is:


# Agent
#  ↓
# Model update
#  ↓
# Tool call update
#  ↓
# Tool result update
#  ↓
# Model update
#  ↓
# Final update



# This is particularly useful in UI applications where you want to show:



# Thinking...
# Using weather tool...
# Got weather data...
# Generating answer...



# However, for production applications you should decide carefully which internal events 
# should actually be shown to the user rather than exposing everything.

### 9. Agent + Structured Output

In [18]:
# You already learned structured output.

# Now combine it with agents.

# Suppose your application wants:

# {
#     "answer": "...",
#     "category": "weather",
#     "confidence": 0.95
# }



# Create a schema
from pydantic import BaseModel, Field


class AgentResponse(BaseModel):
    answer: str
    category: str
    confidence: float = Field(ge=0, le=1)


# create agent
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
    response_format=AgentResponse
)


# Invoke agenbt
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What's the weather in Bhubaneswar?"
        }
    ]
})

# Get the structured result
response = result["structured_response"]

print(response.answer)
print(response.category)
print(response.confidence)




# So:

# Agent
#  ↓
# Tools
#  ↓
# Final result
#  ↓
# Structured schema

# This is extremely useful when the agent is part of a larger application/API.

It's sunny in Bhubaneswar.
weather
1.0


### 10. Agent + RAG

In [ ]:
# Now combine Agents + Tools + RAG.

# A normal RAG system is:

# User
#  ↓
# Query
#  ↓
# Retriever
#  ↓
# Documents
#  ↓
# Prompt
#  ↓
# Gemini
#  ↓
# Answer

# The retriever is always part of the workflow.



# Agentic RAG

# You can expose your knowledge base as a tool:

# User
#  ↓
# Agent
#  ↓
# "Do I need the knowledge base?"
#  ↓
# Yes
#  ↓
# Retriever / Search Tool
#  ↓
# Documents
#  ↓
# Gemini
#  ↓
# Answer





# For example:
from langchain_core.tools import tool

@tool
def search_knowledge_base(query: str) -> str:
    """Search the company's knowledge base for relevant information."""
    # Your vector database / retriever would be used here.
    return "Relevant company documentation..."



# Then:

agent = create_agent(
    model=llm,
    tools=[search_knowledge_base],
    system_prompt=(
        "Answer questions using the knowledge base "
        "when company-specific information is required."
    )
)

# Now the agent can decide whether to use the knowledge base.

### 11. Agent Error Handling

In [ ]:
# Real tools can fail.



# For example:

# API unavailable
# Database timeout
# Invalid input
# Authentication failure
# Missing parameter

# A tool should handle expected errors.



# Example:

@tool
def divide(a: float, b: float) -> str:
    """Divide a number by another number."""

    if b == 0:
        return "Error: division by zero is not allowed."

    return str(a / b)

# Now:

divide.invoke({
    "a": 10,
    "b": 0
})



# returns:

# Error: division by zero is not allowed.

# instead of throwing an uncontrolled error.

# The agent can then receive the result and respond appropriately.

### 12. Retry and Fallback

In [ ]:
# For external services, you may need:

# Agent
#  ↓
# Tool
#  ↓
# Failure
#  ↓
# Retry
#  ↓
# Success

# or:

# Primary tool
#  ↓
# Failure
#  ↓
# Fallback tool
#  ↓
# Result





# For example:

# Primary weather API
#         ↓
#       failed
#         ↓
# Backup weather API
#         ↓
#       result

# This is more of a production engineering concern than an introductory LangChain concept, so learn the idea first.

### 13. Human-in-the-Loop

In [ ]:
# This becomes important when an agent can perform real-world actions.

# Imagine:

@tool
def send_email(to: str, message: str) -> str:
    """Send an email."""
    ...




# You probably don't want:

# Agent
#  ↓
# Decides to send email
#  ↓
# Immediately sends email




# A safer workflow is:

# Agent
#  ↓
# Decides to send email
#  ↓
# Pause
#  ↓
# Human approval
#  ↓
# YES
#  ↓
# send_email()



# This is called human-in-the-loop.

# It is especially useful for:

# Send email
# Delete data
# Modify database
# Make purchases
# Deploy code
# Create appointments

# For read-only operations, approval may not be necessary.

### 14. Tool Permissions

In [ ]:
# An agent should only receive the tools it actually needs.

# Bad design:

agent = create_agent(
    model=llm,
    tools=[
        search,
        database,
        delete_data,
        send_email,
        payment,
        deploy
    ]
)

# if the agent only needs to answer questions.

# Better:

agent = create_agent(
    model=llm,
    tools=[
        search
    ]
)


# Think of tools as permissions.


# Read tools
#     ↓
# Lower risk

# Write tools
#     ↓
# Higher risk

# Destructive tools
#     ↓
# Highest risk

# This becomes important when building production agents.

### 15. ReAct — Do You Need It?

In [ ]:
# You will see ReAct in many older tutorials.



# The basic idea is:

# Reason
#  ↓
# Act
#  ↓
# Observe
#  ↓
# Reason
#  ↓
# Act
#  ↓
# Observe



# For example:

# Question
#  ↓
# Decide weather information is needed
#  ↓
# Call weather tool
#  ↓
# Observe weather result
#  ↓
# Generate answer




# The important concept is the reason → action → observation loop.

# You should understand this concept.

# But you don't need to spend a lot of time implementing an old ReAct agent manually.

# Modern agent APIs abstract much of this process.

### 16. Agent + Database

In [ ]:
# This is a very practical application.



# Suppose your application has:

# customers
# orders
# products


# You can expose controlled operations as tools.



@tool
def get_customer_orders(customer_id: int) -> str:
    """Get orders for a customer."""
    # Query your database here.
    return "Order #1024: Laptop"



# Then:

# User
#  ↓
# Agent
#  ↓
# get_customer_orders()
#  ↓
# Database
#  ↓
# Orders
#  ↓
# Gemini
#  ↓
# Answer



# Instead of giving Gemini unrestricted database access, you expose specific operations.

# That's much easier to control.

### 17. Agent + External APIs

In [ ]:
# Same concept:

# Agent
#  ├── Weather API
#  ├── Search API
#  ├── Payment API
#  ├── CRM API
#  └── Internal API

# Each capability can become a tool.

# Example:

@tool
def get_stock_price(symbol: str) -> str:
    """Get the current stock price for a stock symbol."""
    # Call your stock API here.
    return "..."

# The agent can decide whether that capability is relevant.

### 18. Agent + Routing

In [ ]:
# You already learned RunnableBranch.

# Now compare that with agent routing.




# Runnable routing



# You explicitly define:

# If A → Chain A
# If B → Chain B
# Otherwise → Chain C



# Example:

# RunnableBranch(
#     (condition_a, chain_a),
#     (condition_b, chain_b),
#     chain_c
# )





# Agent routing



# You give the model tools:

# Weather tool
# Calculator tool
# RAG tool
# Database tool



# and let the model decide:

# "What should I use?"


# So:

# RunnableBranch
#     ↓
# Developer-defined routing


# Agent
#     ↓
# Model-driven routing


# This is an important distinction.

### 19. When Should You Use a Chain vs Agent?

In [ ]:
# Don't automatically use an agent for everything.



# Use a Chain when:
# Workflow is predictable
#        ↓
# Step 1 → Step 2 → Step 3



# Example:

# PDF
#  ↓
# Retrieve documents
#  ↓
# Prompt
#  ↓
# Gemini
#  ↓
# Answer

# A normal RAG chain is often enough.




# Use an Agent when:
# Workflow is unpredictable
#        ↓
# Different tools may be needed
#        ↓
# Order of actions can change



# Example:

# User request
#  ↓
# Maybe search
#  ↓
# Maybe database
#  ↓
# Maybe calculator
#  ↓
# Maybe API
#  ↓
# Final answer